In [19]:
import pandas as pd
from pathlib import Path
import datetime

# Task 1: Target Selection & Scraping

> 1.1 Data Selection: Choose one disaster category (Earthquakes, Tsunamis, Floods, Cyclones, Volcanoes, Droughts, or Forest Fires) and choose two different regions.




1.1: Choosing ***Floods*** because of their:
* High data availability,
* High frequency,
* Predictability (unlike earthquakes/volcanoes which begin underground, floods are surface-level phenomena that are easier to measure and observe reliably)
* Localisation to coastal and river basin regions

> 1.2 The "Source of Truth" Scraper: Identify two specific events from those two regions:
> * Event A: A "Historical" event (at least 5–8 years old).
> * Event B: A "Recent" event (within the last 12 months).

1.2: Choosing ***Italy*** and ***Bangladesh*** <br>
in ranges 24-Feb-2026 to 24-Feb-2025 (12 months, "Recent") and 24-Feb-2026 to 24-Feb-2018 (8 years, Historical)

* **Frequency disparity:** Bangladesh is the "flood capital of the world", facing annual, large-scale inundation, while Italy faces frequent, but far more localised, intense, and sudden floods.

* **Vulnerability:** Bangladesh ranks higher in global flood risk, with 3.5 million people at risk of river flooding annually.

* **Human Impact:** Bangladesh experiences far higher death tolls and displacement from floods compared to Italy.

* **Economy:** Bangladesh is a developing country with far less resources to spend on disaster management as opposed to Italy.

---

References:
1. https://unece.org/sites/default/files/2025-04/03_05_Kazi%20Shahidur%20Rahman_14.03.2025.pdf
2. https://floodlist.com/europe/weather-extremes-italy-100-municipalities-2010-2015#:~:text=The%20Legambiente%20report%20says%20that,climate%20change%20in%20the%20country.

In [1]:
import requests

URL = "https://www.gdacs.org/Alerts/default.aspx"
page = requests.get(URL)

print(page.text)



<!DOCTYPE html>

<html xmlns="http://www.w3.org/1999/xhtml">
<head><title>
	GDACS - Global Disaster Awareness and Coordination System
</title>
    <link rel="stylesheet" href="../scripts/openlayers/3.20.1/ol.css" type="text/css" />
    <link rel="stylesheet" href="../scripts/openlayers/layerswitcher/ol3-layerswitcher.css" />
    <link href="../Styles/gdacs_mapstyle.css" rel="stylesheet" />
    <link rel="stylesheet" href="../Content/all.css" />
    <script type="text/javascript" src="../scripts/openlayers/3.20.1/ol.js"></script>
    <script type="text/javascript" src="../scripts/openlayers/layerswitcher/ol3-layerswitcher.js"></script>
    <script type="text/javascript" src="../Scripts/Search/searchtools.js"></script>
    <link href="../Styles/gdacs_2.0_bootstrap4to5.css" rel="stylesheet" />

<link href="/Content/bootstrap.min.css" rel="stylesheet" /><link href="/Styles/gdacs_2.0.css" rel="stylesheet" /><meta name="viewport" content="width=device-width, initial-scale=1.0" /><link rel=

The page elements are clearly rendered dynamically using JavaScript, so using *Selenium*.

## Source of Truth Scraper

In [1]:
from selenium import webdriver
from selenium.webdriver.common.by import By # to locate elements by ID/class
from selenium.webdriver.common.keys import Keys # typing input
from selenium.webdriver.support.ui import WebDriverWait, Select
from selenium.webdriver.support import expected_conditions as EC

import random, time 

options = webdriver.SafariOptions()
options.add_argument("--headless") # doesn't pop up browser window, runs in bg.

driver = webdriver.Safari(options=options)
driver.get("https://www.gdacs.org/Alerts/default.aspx")
wait = WebDriverWait(driver, 10)

driver.implicitly_wait(10) # wait for the page to load

# Accept cookies
# (CSS selector is of the form class1.class2.class3, so using that for composite class name selection)
acceptEssentialCookies = driver.find_element(By.CSS_SELECTOR, ".wt-ecl-button.wt-ecl-button--primary.wt-cck--actions-button")
acceptEssentialCookies.click()

# FILLING THE FORM
# 1. FLOOD CATEGORY (id = inputChFl)
floodCheckbox = driver.find_element(By.ID, "inputChFl")
floodCheckbox.click()
print("Selected floods")

# 2. Fill FROM date (id="inputDateFrom")
inputDateFROM = driver.find_element(By.ID, "inputDateFrom")
FromDate = "2025-02-24" # one year ago

inputDateFROM.clear() # default date cleared
inputDateFROM.send_keys(FromDate);

# 3. Fill TO date (id="inputDateTo")
inputDateTO = driver.find_element(By.ID, "inputDateTo")
ToDate = "2026-02-24" # today

inputDateTO.clear() # default date cleared
inputDateTO.send_keys(ToDate);

# 4. id=inputAlert dropdown alert level
dropdown_element = driver.find_element(By.ID, "inputAlert")

# using JS to set the value of the dropdown 
# & triggering the change event
driver.execute_script("""
    arguments[0].value = "Green;Orange;Red";
    arguments[0].dispatchEvent(new Event('change'));
""", dropdown_element)

print("Selected All for Alert Level")

# 5. country (id="inputCountry")
country_input = driver.find_element(By.ID, "inputCountry")
country = "Bangladesh"
country_input.send_keys(country)

# 6. click on the Search button (id="btnsearch")
search_button = driver.find_element(By.ID, "btnsearch")
search_button.click()

# 7. from given result "clickable rows"
# pick one at random
# check its link (data-href attribute), print it and then nav to it

# wait until rows are loaded
rows = wait.until(
    EC.presence_of_all_elements_located((By.CSS_SELECTOR, "tr.clickable-row"))
)

print("Total rows found:", len(rows))

# pick a row at random
random_row = random.choice(rows)

# extract link from data-href attribute and print it
# next scraper will take it from there
random_link = random_row.get_attribute("data-href")

print("Random link: for " + country + ":", random_link)

driver.quit()

Selected floods
Selected All for Alert Level
Total rows found: 100
Random link: for Bangladesh: https://www.gdacs.org/report.aspx?eventid=1001199&episodeid=13&eventtype=TC


Learning notes from building this scraper:
* Instead of treating the dropdown as a set of clickable buttons, one needs to use the Select class. 
* Dropdown was especially difficult to work with as it is a JavaScript (dynamic) element that can't directly be interacted with, even after wait times. Tried using a select by value, visible text and index, and none of them directly worked.
* Incorporated wait times in case it was a DOM loading issue
* Finally, realised that it needed a direct JS script since this was a dynamic element.

Refs: https://realpython.com/modern-web-automation-with-python-and-selenium/,  https://www.selenium.dev, https://www.selenium.dev/selenium/docs/api/java/org/openqa/selenium/safari/SafariDriver.html

Make linked list of "random links" collected, to be used by the next scraper

Would want to make this scraper **more modular and generalisable** so that I can call it on different countries, to and from dates (strings) and checkboxes.

In [ ]:
def scrape_gdacs(country, date_from, date_to, headless=True):
    from selenium import webdriver
    from selenium.webdriver.common.by import By
    from selenium.webdriver.common.keys import Keys
    from selenium.webdriver.support.ui import WebDriverWait
    from selenium.webdriver.support import expected_conditions as EC
    import random, time
    
    options = webdriver.SafariOptions()
    if headless:
        options.add_argument("--headless")
    
    driver = webdriver.Safari(options=options)
    wait = WebDriverWait(driver, 10)
    
    try:
        driver.get("https://www.gdacs.org/Alerts/default.aspx")
        driver.implicitly_wait(10)
        
        # Accept cookies
        acceptEssentialCookies = driver.find_element(By.CSS_SELECTOR, ".wt-ecl-button.wt-ecl-button--primary.wt-cck--actions-button")
        acceptEssentialCookies.click()
        
        # 1. FLOOD CATEGORY (id = inputChFl)
        floodCheckbox = driver.find_element(By.ID, "inputChFl")
        floodCheckbox.click()
        
        # 2. Fill FROM date (id="inputDateFrom")
        inputDateFROM = driver.find_element(By.ID, "inputDateFrom")
        inputDateFROM.clear()
        inputDateFROM.send_keys(str(date_from))
        
        # 3. Fill TO date (id="inputDateTo")
        inputDateTO = driver.find_element(By.ID, "inputDateTo")
        inputDateTO.clear()
        inputDateTO.send_keys(str(date_to))
        
        # 4. id=inputAlert dropdown alert level
        dropdown_element = driver.find_element(By.ID, "inputAlert")
        driver.execute_script("""
            arguments[0].value = "Green;Orange;Red";
            arguments[0].dispatchEvent(new Event('change'));
        """, dropdown_element)
        
        # 5. country (id="inputCountry")
        country_input = driver.find_element(By.ID, "inputCountry")
        country_input.send_keys(country)
        country_input.send_keys(Keys.ENTER)  # Confirm country selection
        
        # 6. click on the Search button (id="btnsearch")
        search_button = driver.find_element(By.ID, "btnsearch")
        search_button.click()
        
        # 7. from given result "clickable rows"
        # pick one at random
        # check its link (data-href attribute), print it and then nav to it
        
        # wait until rows are loaded
        rows = wait.until(
            EC.presence_of_all_elements_located((By.CSS_SELECTOR, "tr.clickable-row"))
        )
        
        # pick a row at random
        random_row = random.choice(rows)
        
        # extract link from data-href attribute and print it
        random_link = random_row.get_attribute("data-href")
        
        result = {
            "country": country,
            "date_from": date_from,
            "date_to": date_to,
            "total_results": len(rows),
            "random_link": random_link
        }
        
        return result
    finally:
        driver.quit()

In [ ]:
jobs = [
    ("inputChFl", "Bangladesh", "2025-02-25", "2026-02-25"), # recent (2025-2026)
    ("inputChFl", "Bangladesh", "2018-02-25", "2024-02-25"), # historical (2018-2024)
    ("inputChFl", "Italy", "2025-02-25", "2026-02-25"), # recent 
    ("inputChFl", "Italy", "2018-02-25", "2024-02-25"), # historical
]

In [ ]:
all_results = []

for _, country, date_from, date_to in jobs:
    result = scrape_gdacs(country, date_from, date_to, headless=True)
    all_results.append(result)

print(all_results)

In [ ]:
import pandas as pd

df = pd.DataFrame(all_results)
df

Having validated the working of this scraper and seeing that it is correctly extracting links, sending this to the deep tab extraction scraper, and saving it as a csv permanently for further use:

In [ ]:
from pathlib import Path
from datetime import datetime

save_dir = Path("data")
save_dir.mkdir(exist_ok=True)

latest_csv_path = save_dir / "gdacs_df_latest.csv"
versioned_csv_path = save_dir / f"gdacs_df_{datetime.now():%Y%m%d_%H%M%S}.csv"

df.to_csv(latest_csv_path, index=False)
df.to_csv(versioned_csv_path, index=False)

print("Saved latest:", latest_csv_path)
print("Saved versioned:", versioned_csv_path)

# INBUILT VERSIONING APPROACH: save with timestamped filename, and also overwrite a "latest" file for easy access.
# Note: to reload later in any new session, have to do:
# import pandas as pd
# df = pd.read_csv("data/gdacs_df_latest.csv")

## Deep Tab Extraction Scraper

### Additional insights I'm extracting from each tab:
- Event summary table fields: GDACS ID, deaths, displaced, and event date range (From-To).
- GDACS score panel: numeric score value shown in the gauge and model info link.
- Impact context signals: presence of exposed-population section and available map/layer context.
- Activation/operations indicators: Virtual OSOCC, Meteo assessment, Satellite products, Analytical products.
- Media context: casualties mention count and last-hour article count from the media summary line.
- Description/meta: headline title, bulletin date, provider/source link.
- External data-source inventory from resources block (e.g., EC-JRC, NASA, WMO, INFORM counts).

In [3]:
import pandas as pd
df = pd.read_csv("data/gdacs_df_20260225_215447.csv")

Having successfully run the scraper on the site a few times, now choosing the 20260225_215447 version of the output (CSV file) to run the advanced deep tab scraper.

In [4]:
df.head() # sanity check

,country,date_from,date_to,total_results,random_link
0,Bangladesh,2025-02-25,2026-02-25,2,https://www.gdacs.org/report.aspx?eventid=1103...
1,Bangladesh,2018-02-25,2024-02-25,18,https://www.gdacs.org/report.aspx?eventid=1100...
2,Italy,2025-02-25,2026-02-25,15,https://www.gdacs.org/report.aspx?eventid=1103...
3,Italy,2018-02-25,2024-02-25,54,https://www.gdacs.org/report.aspx?eventid=1101...


### Summary Tab

In [ ]:
import json
import re
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

options = webdriver.SafariOptions()
options.add_argument("--headless") # doesn't pop up browser window, runs in bg.

driver = webdriver.Safari(options=options)
driver.get("https://www.gdacs.org/Alerts/default.aspx")
wait = WebDriverWait(driver, 12)


def _extract_int(text):
    if not text:
        return None
    m = re.search(r"\d+", text.replace(",", ""))
    return int(m.group()) if m else None


def _safe_text(driver, selector):
    try:
        return driver.find_element(By.CSS_SELECTOR, selector).text.strip()
    except Exception:
        return ""


def _parse_news_text(news_text):
    # Expected pattern is "M/D/YYYY H:MM:SS AM" 
    date_str = None
    time_str = None
    first_paragraph = ""

    match = re.search(r"(\d{1,2}/\d{1,2}/\d{4})\s+(\d{1,2}:\d{2}:\d{2}\s*[AP]M)", news_text)
    if match:
        date_str = match.group(1)
        time_str = match.group(2).upper().replace(" ", "")
        first_paragraph = news_text[match.end():].strip(" .\n")
    else:
        first_paragraph = news_text.strip()

    return date_str, time_str, first_paragraph


def scrape_gdacs_deep_tabs(report_link, headless=True):
    options = webdriver.SafariOptions()
    if headless:
        options.add_argument("--headless")

    driver = webdriver.Safari(options=options)
    wait = WebDriverWait(driver, 12)

    try:
        driver.get(report_link)
        wait.until(EC.presence_of_element_located((By.TAG_NAME, "body")))

        # Extract eventid from the URL
        url_eventid_match = re.search(r"eventid=(\d+)", report_link)
        url_eventid = url_eventid_match.group(1) if url_eventid_match else None
        print(f"URL eventid: {url_eventid}")

        data = {
            "link": report_link,
            "gdacs_id": None,
            "death_count": None,
            "displaced_count": None,
            "countries": None,
            "event_from_to": None,
            "articles_count": None,
            "about_casualties_count": None,
            "last_hour_count": None,
            "news_items": None,
            "echo_daily_flash": None
        }

        # SUMMARY TABLE
        rows = driver.find_elements(By.CSS_SELECTOR, "#alert_summary_left table.summary tr")
        if not rows:
            rows = driver.find_elements(By.CSS_SELECTOR, "table.summary tr")

        row_map = {}
        for row in rows:
            cells = row.find_elements(By.TAG_NAME, "td")
            if len(cells) >= 2:
                key = cells[0].text.strip().lower().replace(":", "")
                value = cells[1].text.strip()
                row_map[key] = value

        data["gdacs_id"] = row_map.get("gdacs id")
        print(f"Extracted gdacs_id: {data['gdacs_id']}")

        # Compare extracted gdacs_id with URL eventid
        if url_eventid and data["gdacs_id"] != url_eventid:
            print(f"Mismatch detected: URL eventid ({url_eventid}) vs Extracted gdacs_id ({data['gdacs_id']})")

        death_raw = row_map.get("death")
        if death_raw and death_raw != "-":
            data["death_count"] = _extract_int(death_raw)

        displaced_raw = row_map.get("displaced")
        if displaced_raw and displaced_raw != "-":
            data["displaced_count"] = _extract_int(displaced_raw)

        # LATEST MEDIA HEADLILNES SECTION
        summary_news_text = _safe_text(driver, "#summaryNews")
        if summary_news_text:
            articles_match = re.search(r"articles\s*:\s*(\d+)", summary_news_text, flags=re.IGNORECASE)
            if articles_match:
                data["articles_count"] = int(articles_match.group(1))

            casualties_match = re.search(r"about\s+casualties\s*:\s*(\d+)", summary_news_text, flags=re.IGNORECASE)
            if casualties_match:
                data["about_casualties_count"] = int(casualties_match.group(1))

            last_hour_match = re.search(r"last\s+hour\s*:\s*(\d+)", summary_news_text, flags=re.IGNORECASE)
            if last_hour_match:
                data["last_hour_count"] = int(last_hour_match.group(1))

        # FOR EACH ITEM IN NEWS HEADLINES, EXTRACT TITLE, LINK, DATE, TIME, FIRST PARAGRAPH
        news_items = []
        news_boxes = driver.find_elements(By.CSS_SELECTOR, "#news_main .news_box")
        for box in news_boxes:
            try:
                title_el = box.find_element(By.CSS_SELECTOR, ".news_title a")
                title = title_el.text.strip()
                link = title_el.get_attribute("href")
            except Exception:
                # If no anchor, fallback to span
                try:
                    title = box.find_element(By.CSS_SELECTOR, ".news_title").text.strip()
                    link = ""
                except Exception:
                    title = ""
                    link = ""

            try:
                text_el = box.find_element(By.CSS_SELECTOR, ".news_text")
                news_text = text_el.text
            except Exception:
                news_text = ""

            date_str, time_str, first_paragraph = _parse_news_text(news_text)
            if title or link or news_text:
                news_items.append({
                    "title": title,
                    "link": link,
                    "date": date_str,
                    "time": time_str,
                    "first_paragraph": first_paragraph
                })

            # ECHO DAILY FLASH: If the news_title contains 'ECHO' or 'Daily Flash', save the full HTML
            if 'ECHO' in title or 'Daily Flash' in title:
                try:
                    # Save the outer HTML of the news_box
                    data["echo_daily_flash"] = box.get_attribute("outerHTML")
                except Exception:
                    data["echo_daily_flash"] = None

        data["news_items"] = json.dumps(news_items, ensure_ascii=True)
        return data
    finally:
        driver.quit()

# FOR LATER USE:
# deep_data = scrape_gdacs_deep_tabs(df.loc[0, "random_link"], headless=True)
# print(deep_data)

In [9]:
deep_data_list = []
for idx, row in df.iterrows():
    link = row.get("random_link")
    if isinstance(link, str) and link.strip():
        deep_data = scrape_gdacs_deep_tabs(link.strip(), headless=True)
        deep_data_list.append(deep_data)
    else:
        deep_data_list.append({"link": link})
deep_data_list

[{'link': 'https://www.gdacs.org/report.aspx?eventid=1103509&episodeid=1&eventtype=FL',
  'gdacs_id': 'FL 1103509',
  'death_count': 2,
  'displaced_count': None,
  'countries': 'Bangladesh',
  'event_from_to': '18 Sep - 20 Sep',
  'articles_count': 10,
  'about_casualties_count': 0,
  'last_hour_count': 0,
  'news_items': '[{"title": "Ganga Basin Faced Severe Drying Over The Last 30 Years Due To Frequent..", "link": "https://www.news18.com/india/ganga-basin-faced-severe-drying-over-the-last-30-years-due-to-frequent-droughts-shows-new-study-ws-l-9591630.html", "date": "9/23/2025", "time": "11:06:00AM", "first_paragraph": "The Ganga River Basin, lifeline of over 600 million people, experienced its most severe drying trend from 1991 to 2020 \\u2013 the worst in the past 1,000 years, according to a new research, which examined 1,300 years of streamflow data and the i\\t\\t\\t ...(more)\\t\\t"}, {"title": "Saudi Arabia and Pakistan just formed a defence pact\\u2014India must rethin..", "li

The ID's match up perfectly with the initial scraped events in (data/gdacs_df_20260225_215447.csv). I also manually checked them to ensure all the data is accurately scraped!

The following cell was used for versioning to test the scrapers, but need not be used now:

In [ ]:
"""import pandas as pd
from pathlib import Path
from datetime import datetime

# Use the already computed deep_data_list
deep_df = pd.DataFrame(deep_data_list)

# Save to a NEW folder with versioning control
output_dir = Path("deep_tab_outputs")
output_dir.mkdir(exist_ok=True)

enriched_latest = output_dir / "gdacs_deep_tab_latest.csv"
enriched_versioned = output_dir / f"gdacs_deep_tab_{datetime.now():%Y%m%d_%H%M%S}.csv"

deep_df.to_csv(enriched_latest, index=False)
deep_df.to_csv(enriched_versioned, index=False)

print("Saved latest:", enriched_latest)
print("Saved versioned:", enriched_versioned)
deep_df.head()"""

In [20]:
deep_df = pd.DataFrame(deep_data_list)
deep_df.to_csv("deep_tab_outputs/summary.csv", index=False)

In [15]:
# ANOTHER SANITY CHECK (untruncated links)
with pd.option_context('display.max_rows', None, 'display.max_columns', None, 'display.max_colwidth', None):
    display(deep_df["link"])

0    https://www.gdacs.org/report.aspx?eventid=1103509&episodeid=1&eventtype=FL
1    https://www.gdacs.org/report.aspx?eventid=1100273&episodeid=1&eventtype=FL
2    https://www.gdacs.org/report.aspx?eventid=1103220&episodeid=9&eventtype=FL
3    https://www.gdacs.org/report.aspx?eventid=1101050&episodeid=2&eventtype=FL
Name: link, dtype: object

In [17]:
# Compactly viewing the deep_df
priority_cols = [
    "country",
    "report_country",
    "date_from",
    "date_to",
    "random_link",
    "magnitude",
    "alert_level",
    "gdacs_score",
    "gdacs_id",
    "death_count",
    "displaced_count",
    "population_exposed_100km",
    "vulnerability_inform_index",
    "secondary_risks",
    "news_articles_count",
    "about_casualties_count",
    "last_hour_count",
    "social_media_mentions_count",
    "activation_tabs",
    "description_title"
    ]

report_cols = [c for c in priority_cols if c in df.columns]
report_df = df[report_cols].copy()

sort_cols = [c for c in ["country", "report_country", "date_from"] if c in report_df.columns]
if sort_cols:
    report_df = report_df.sort_values(sort_cols).reset_index(drop=True)

print("Reporting columns:", report_cols)
report_df

Reporting columns: ['country', 'date_from', 'date_to', 'random_link']


Reporting columns: ['country', 'date_from', 'date_to', 'random_link']


,country,date_from,date_to,random_link
0,Bangladesh,2018-02-25,2024-02-25,https://www.gdacs.org/report.aspx?eventid=1100...
1,Bangladesh,2025-02-25,2026-02-25,https://www.gdacs.org/report.aspx?eventid=1103...
2,Italy,2018-02-25,2024-02-25,https://www.gdacs.org/report.aspx?eventid=1101...
3,Italy,2025-02-25,2026-02-25,https://www.gdacs.org/report.aspx?eventid=1103...


> * 1.3 Deep-Tab Extraction: Your scraper must navigate to the specific report pages and extract:
> 1. Summary Tab: Magnitude, Alert Level (Green/Orange/Red), and Country.
> 2. Impact Tab: Population exposed (e.g., people in 100km buffer), Vulnerability
> score (INFORM index), and secondary risks.
> 3. Media Tab: Total number of news articles and social media mention counts.

### Media Tab

The summary tab had opened automatically with the given URL. Now one must specify explicitly to navigate to the menu to access the other tabs: Media and Impact.

In [78]:
# Define a function to scrape media tab for a single event
def scrape_media_for_event(gdacs_id, driver, wait):
    if not gdacs_id:
        print("Invalid gdacs_id provided.")
        return {"gdacs_id": gdacs_id, "media_articles": []}
    try:
        # Robustly extract eventid and eventtype
        if isinstance(gdacs_id, str) and " " in gdacs_id:
            eventtype, eventid = gdacs_id.split()
        else:
            eventtype = "FL"
            eventid = str(gdacs_id)
        # Compose the media tab URL for debugging
        media_url = f"https://www.gdacs.org/media.aspx?eventid={eventid}&episodeid=1&eventtype={eventtype}"
        print(f"DEBUG: Would scrape media tab at: {media_url}")
        driver.get(media_url)
        wait.until(EC.presence_of_element_located((By.TAG_NAME, "body")))
        articles = []
        # Find all media_news_box divs
        news_boxes = driver.find_elements(By.CSS_SELECTOR, "#emmList .media_news_box")
        print(f"Found {len(news_boxes)} media_news_box elements")
        for box in news_boxes:
            try:
                title_el = box.find_element(By.CSS_SELECTOR, ".p_media_title a")
                title = title_el.text.strip()
                link = title_el.get_attribute("href")
            except Exception as e:
                print(f"Error extracting title/link: {e}")
                title = ""
                link = ""
            try:
                date_source = box.find_element(By.CSS_SELECTOR, ".p_media_dida").text.strip()
            except Exception as e:
                print(f"Error extracting date_source: {e}")
                date_source = ""
            try:
                summary = box.find_element(By.CSS_SELECTOR, ".p_summary").text.strip()
            except Exception as e:
                print(f"Error extracting summary: {e}")
                summary = ""
            articles.append({
                "title": title,
                "link": link,
                "date_source": date_source,
                "summary": summary
            })
        return {"gdacs_id": gdacs_id, "media_articles": articles}
    except Exception as e:
        print(f"Error processing gdacs_id {gdacs_id}: {e}")
        return {"gdacs_id": gdacs_id, "media_articles": []}

# Main script to process all rows in summary.csv
import pandas as pd
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from pathlib import Path

# Load summary.csv
input_file = "deep_tab_outputs/summary.csv"
try:
    summary_df = pd.read_csv(input_file)
    if "gdacs_id" not in summary_df.columns:
        raise ValueError("Input file does not contain the required 'gdacs_id' column.")
    print("Loaded summary DataFrame:")
    print(summary_df.head())
except FileNotFoundError:
    print(f"Error: Input file '{input_file}' not found.")
    raise
except Exception as e:
    print(f"Error loading input file: {e}")
    raise

media_results = []

# Initialize WebDriver once for all events
options = webdriver.SafariOptions()
options.add_argument("--headless")
driver = webdriver.Safari(options=options)
wait = WebDriverWait(driver, 20)  # Increased timeout to 20 seconds

try:
    for idx, row in summary_df.iterrows():
        gdacs_id = row.get("gdacs_id")
        print(f"Processing row {idx+1}/{len(summary_df)}: gdacs_id: {gdacs_id}")
        try:
            result = scrape_media_for_event(gdacs_id, driver, wait)
        except Exception as e:
            print(f"Exception for row {idx+1}: {e}")
            result = {"gdacs_id": gdacs_id, "media_articles": []}
        media_results.append(result)
finally:
    driver.quit()

# Convert to DataFrame for analysis
media_flat = []
for event in media_results:
    event_id = event["gdacs_id"]
    for article in event["media_articles"]:
        media_flat.append({"gdacs_id": event_id, **article})
media_df = pd.DataFrame(media_flat)

print("Scraped media_df shape:", media_df.shape)
print(media_df.head())

# Save to CSV for further analysis
output_dir = Path("deep_tab_outputs")
output_dir.mkdir(exist_ok=True)
media_csv = output_dir / "media.csv"
media_df.to_csv(media_csv, index=False)
print("Saved media tab to:", media_csv)

Loaded summary DataFrame:
                                                link    gdacs_id  death_count  \
0  https://www.gdacs.org/report.aspx?eventid=1103...  FL 1103509            2   
1  https://www.gdacs.org/report.aspx?eventid=1100...  FL 1100541           67   
2  https://www.gdacs.org/report.aspx?eventid=1103...  FL 1103166            0   
3  https://www.gdacs.org/report.aspx?eventid=1101...  FL 1101944            0   

   displaced_count   countries    event_from_to  articles_count  \
0              NaN  Bangladesh  18 Sep - 20 Sep              10   
1          56000.0  Bangladesh  28 Jun - 10 Jul              68   
2           1084.0       Italy  14 Mar - 30 Mar              40   
3             48.0       Italy  15 Apr - 17 Apr               9   

   about_casualties_count  last_hour_count  \
0                       0                0   
1                      16                0   
2                       0                0   
3                       0                0   

 

In [77]:
# DEBUG: Print all media_results and resulting DataFrame
import pandas as pd
from pathlib import Path

print('media_results:')
for event in media_results:
    print(f"gdacs_id: {event['gdacs_id']}, articles: {len(event['media_articles'])}")
    for article in event['media_articles']:
        print(f"  - {article['title'][:60]} ...")

print(f"\nmedia_flat (length): {len(media_flat)}")
print(f"media_df shape: {media_df.shape}")
print(media_df.groupby('gdacs_id').size())

media_csv = Path("deep_tab_outputs/media.csv")
if media_csv.exists():
    print("\nmedia.csv exists. First few rows:")
    print(pd.read_csv(media_csv).head(10))
else:
    print("media.csv does not exist!")


media_results:
gdacs_id: FL 1103509, articles: 10
  - Breakthrough in cold-tolerant rice development benefits haor ...
  - Ganga Basin Faced Severe Drying Over The Last 30 Years Due T ...
  - Saudi Arabia and Pakistan just formed a defence pact—India m ...
  - Signals lowered at seaports, fresh low brews ...
  - CM meets envoys of Turkiye, Bangladesh ...
  - Sindh CM, Turkish and Bangladeshi envoys stress trade cooper ...
  - WFP Bangladesh Country Brief, August 2025 ...
  - WFP Bangladesh Country Brief, August 2025 ...
  - CM, Turkish CG, Bangladesh Deputy HC, stress trade, investme ...
  - CM, Turkish CG, Bangladesh Deputy HC, stress trade, investme ...
gdacs_id: FL 1100541, articles: 0
gdacs_id: FL 1103166, articles: 0
gdacs_id: FL 1101944, articles: 0

media_flat (length): 10
media_df shape: (10, 5)
gdacs_id
FL 1103509    10
dtype: int64

media.csv exists. First few rows:
     gdacs_id                                              title  \
0  FL 1103509  Breakthrough in cold-toleran

In [71]:
# Diagnostics: Check if all events were processed and file was written
import pandas as pd
from pathlib import Path

# Check media_results and media_df
print(f"Number of events processed (media_results): {len(media_results)}")
print(f"Number of rows in media_df: {len(media_df)}")
print("First few rows of media_df:")
print(media_df.head())

# Check if media.csv exists and print its contents
media_csv = Path("deep_tab_outputs/media.csv")
if media_csv.exists():
    print("media.csv exists. First few rows:")
    print(pd.read_csv(media_csv).head())
else:
    print("media.csv does not exist!")


Number of events processed (media_results): 4
Number of rows in media_df: 10
First few rows of media_df:
     gdacs_id                                              title  \
0  FL 1103509  Breakthrough in cold-tolerant rice development...   
1  FL 1103509  Ganga Basin Faced Severe Drying Over The Last ...   
2  FL 1103509  Saudi Arabia and Pakistan just formed a defenc...   
3  FL 1103509       Signals lowered at seaports, fresh low brews   
4  FL 1103509             CM meets envoys of Turkiye, Bangladesh   

                                                link  \
0  https://www.cgiar.org/news-events/news/breakth...   
1  https://www.news18.com/india/ganga-basin-faced...   
2  https://www.livemint.com/opinion/online-views/...   
3       https://bdnews24.com/bangladesh/b6fb169e7a6a   
4  https://www.brecorder.com/news/40384037/cm-mee...   

                      date_source  \
0      2025-09-23T20:27:00(cgiar)   
1     2025-09-23T11:06:00(news18)   
2   2025-09-23T09:30:00(livemint)   
3

In [21]:
# for each link in data/gdacs_df_20260225_215447.csv

# navigate to Media tab
"""<a class="nav-link " href="/media.aspx?eventid=1103509&amp;episodeid=1&amp;eventtype=FL">Media</a>"""

# extract news information for each media_news_box on the page
"""<div id="contentSectionNews" class="col-lg-12">
                    <h2>All headlines on this Alert</h2>
                    <p class="p_summary">The headlines below have been automatically extracted by the <a href="http://emm.newsbrief.eu" "target="_blank">Europe Media Monitor</a>.</p>
                    <div id="scrollList" class="alert_summary_fifth_b" style="max-height:500px">
                        <div id="emmList"><div class="media_news_box">	<p class="p_media_title"><a href="https://www.cgiar.org/news-events/news/breakthrough-in-cold-tolerant-rice-development-benefits-haor-farmers-under-haor-kgf-project-in-bangladesh">Breakthrough in cold-tolerant rice development benefits haor farmers under Haor-KGF Project in <span style="background-color:Yellow" class="highlight">Bangladesh</span> International Rice Research Institute (IRRI) 17.09.25 Climate adaptation &amp; mitigation By Nusrat Jahan, Muhiuddin Faruquee, and Mohammad Rafiqul Islam <span style="background-color:Yellow" class="highlight">Bangladesh</span>, August 2025 – After fi… Read....</a></p>	<p class="p_media_dida">2025-09-23T20:27:00(cgiar) </p>	<p class="p_summary">By Nusrat Jahan, Muhiuddin Faruquee, and Mohammad Rafiqul Islam. <span style="background-color:Yellow" class="highlight">Bangladesh</span>, August 2025 – After five years of intensive research and collaboration, the Haor-KGF project has reached a major milestone: the development of advanced short-duration, cold-tolerant rice lines that can help farmers in....</p></div><div class="media_news_box">	<p class="p_media_title"><a href="https://www.news18.com/india/ganga-basin-faced-severe-drying-over-the-last-30-years-due-to-frequent-droughts-shows-new-study-ws-l-9591630.html">Ganga Basin Faced Severe Drying Over The Last 30 Years Due To Frequent Droughts, Shows New Study</a></p>	<p class="p_media_dida">2025-09-23T11:06:00(news18) </p>	<p class="p_summary">The Ganga River Basin, lifeline of over 600 million people, experienced its most severe drying trend from 1991 to 2020 – the worst in the past 1,000 years, according to a new research, which examined 1,300 years of streamflow data and the increasing drought frequency amid an erratic monsoon.</p></div><div class="media_news_box">	<p class="p_media_title"><a href="https://www.livemint.com/opinion/online-views/saudi-arabia-pakistan-defence-agreement-smda-nuclear-umbrella-india-saudi-china-middle-east-west-asia-israel-qatar-war-11758521972077.html">Saudi Arabia and Pakistan just formed a defence pact—India must rethink strategy</a></p>	<p class="p_media_dida">2025-09-23T09:30:00(livemint) </p>	<p class="p_summary">This Nato-like defence commitment turns their ‘short-of-alliance partnership’ into a ‘formal bilateral alliance.’ For the KSA, Israel’s recent attack on Qatar and concerns for the Gulf region’s overall security is a major driver; whereas, for Pakistan, any development in its alliance network always had an Indian element as a motivating factor.</p></div><div class="media_news_box">	<p class="p_media_title"><a href="https://bdnews24.com/bangladesh/b6fb169e7a6a">Signals lowered at seaports, fresh low brews</a></p>	<p class="p_media_dida">2025-09-23T07:58:00(bdnews24) </p>	<p class="p_summary">The <span style="background-color:Yellow" class="highlight">Bangladesh</span> Meteorological Department has instructed the seaports of Chattogram, Cox’s Bazar, Mongla and Payra to lower their cautionary signals, saying there is currently no possibility of squally weather in the coastal areas. The directive came in a special bulletin issued by the department on....</p></div><div class="media_news_box">	<p class="p_media_title"><a href="https://www.brecorder.com/news/40384037/cm-meets-envoys-of-turkiye-bangladesh">CM meets envoys of Turkiye, <span style="background-color:Yellow" class="highlight">Bangladesh</span></a></p>	<p class="p_media_dida">2025-09-23T03:02:00(brecorder) </p>	<p class="p_summary">KARACHI: Sindh Chief Minister Syed Murad Ali Shah held separate meetings with the newly appointed Consul General of Turkiye Ergül Kadak, and the newly appointed Deputy High Commissioner of <span style="background-color:Yellow" class="highlight">Bangladesh</span> Muhammad Sakib Sadakat, at CM House. Welcoming both diplomats, the chief minister reaffirmed the....</p></div><div class="media_news_box">	<p class="p_media_title"><a href="https://dailytimes.com.pk/1372661/sindh-cm-turkish-and-bangladeshi-envoys-stress-trade-cooperation/">Sindh CM, Turkish and <span style="background-color:Yellow" class="highlight">Bangladesh</span>i envoys stress trade cooperation</a></p>	<p class="p_media_dida">2025-09-22T23:31:00(dailytimesPK) </p>	<p class="p_summary">Sindh Chief Minister Syed Murad Ali Shah on Tuesday held separate meetings with the newly appointed Consul General of Turkiye, Ergül Kadak and the newly appointed Deputy High Commissioner of <span style="background-color:Yellow" class="highlight">Bangladesh</span>, Muhammad Sakib Sadakat, at CM House. Welcoming both diplomats, the Chief Minister reaffirmed the....</p></div><div class="media_news_box">	<p class="p_media_title"><a href="https://reliefweb.int/report/bangladesh/wfp-bangladesh-country-brief-august-2025">WFP <span style="background-color:Yellow" class="highlight">Bangladesh</span> Country Brief, August 2025</a></p>	<p class="p_media_dida">2025-09-22T21:59:00(reliefWeb) </p>	<p class="p_summary">In Numbers 15,846 mt food distributed; USD 8.2 million distributed in cash-based transfers; USD 46.68 million net funding requirements under the country strategic plan (Sep 2025 – Feb 2026) 1.2 million people assisted* Operational Updates. Emergency preparedness.</p></div><div class="media_news_box">	<p class="p_media_title"><a href="https://reliefweb.int/report/bangladesh/wfp-bangladesh-country-brief-august-2025">WFP <span style="background-color:Yellow" class="highlight">Bangladesh</span> Country Brief, August 2025</a></p>	<p class="p_media_dida">2025-09-22T13:54:00(humanitarianresponse) </p>	<p class="p_summary">In Numbers 15,846 mt food distributed; USD 8.2 million distributed in cash-based transfers; USD 46.68 million net funding requirements under the country strategic plan (Sep 2025 Feb 2026) 1.2 million people assisted* Operational Updates. Emergency preparedness.</p></div><div class="media_news_box">	<p class="p_media_title"><a href="https://www.app.com.pk/domestic/cm-turkish-cg-bangladesh-deputy-hc-stress-trade-investment-cooperation/">CM, Turkish CG, <span style="background-color:Yellow" class="highlight">Bangladesh</span> Deputy HC, stress trade, investment cooperation</a></p>	<p class="p_media_dida">2025-09-22T13:08:00(app-en) </p>	<p class="p_summary">KARACHI, Sep 22 (APP): Sindh Chief Minister Syed Murad Ali Shah on Tuesday held separate meetings with the newly appointed Consul General of Turkiye, Ergül Kadak and the newly appointed Deputy High Commissioner of <span style="background-color:Yellow" class="highlight">Bangladesh</span>, Muhammad Sakib Sadakat, at CM House.</p></div><div class="media_news_box">	<p class="p_media_title"><a href="https://www.urdupoint.com/en/pakistan/cm-turkish-cg-bangladesh-deputy-hc-stress-2048701.html">CM, Turkish CG, <span style="background-color:Yellow" class="highlight">Bangladesh</span> Deputy HC, stress trade, investment cooperation</a></p>	<p class="p_media_dida">2025-09-22T12:48:00(urdupoint) </p>	<p class="p_summary">KARACHI, (UrduPoint / Pakistan Point News - 22nd Sep, 2025) Sindh Chief Minister Syed Murad Ali Shah on Tuesday held separate meetings with the newly appointed Consul General of Turkiye, Ergül Kadak and the newly appointed Deputy High Commissioner of <span style="background-color:Yellow" class="highlight">Bangladesh</span>, Muhammad Sakib Sadakat, at CM House.</p></div></div>
                    </div>
            
                </div>
<h2>All headlines on this Alert</h2>
<p class="p_summary">The headlines below have been automatically extracted by the <a href="http://emm.newsbrief.eu" "target="_blank">Europe Media Monitor</a>.</p>
<div id="scrollList" class="alert_summary_fifth_b" style="max-height:500px">
                        <div id="emmList"><div class="media_news_box">	<p class="p_media_title"><a href="https://www.cgiar.org/news-events/news/breakthrough-in-cold-tolerant-rice-development-benefits-haor-farmers-under-haor-kgf-project-in-bangladesh">Breakthrough in cold-tolerant rice development benefits haor farmers under Haor-KGF Project in <span style="background-color:Yellow" class="highlight">Bangladesh</span> International Rice Research Institute (IRRI) 17.09.25 Climate adaptation &amp; mitigation By Nusrat Jahan, Muhiuddin Faruquee, and Mohammad Rafiqul Islam <span style="background-color:Yellow" class="highlight">Bangladesh</span>, August 2025 – After fi… Read....</a></p>	<p class="p_media_dida">2025-09-23T20:27:00(cgiar) </p>	<p class="p_summary">By Nusrat Jahan, Muhiuddin Faruquee, and Mohammad Rafiqul Islam. <span style="background-color:Yellow" class="highlight">Bangladesh</span>, August 2025 – After five years of intensive research and collaboration, the Haor-KGF project has reached a major milestone: the development of advanced short-duration, cold-tolerant rice lines that can help farmers in....</p></div><div class="media_news_box">	<p class="p_media_title"><a href="https://www.news18.com/india/ganga-basin-faced-severe-drying-over-the-last-30-years-due-to-frequent-droughts-shows-new-study-ws-l-9591630.html">Ganga Basin Faced Severe Drying Over The Last 30 Years Due To Frequent Droughts, Shows New Study</a></p>	<p class="p_media_dida">2025-09-23T11:06:00(news18) </p>	<p class="p_summary">The Ganga River Basin, lifeline of over 600 million people, experienced its most severe drying trend from 1991 to 2020 – the worst in the past 1,000 years, according to a new research, which examined 1,300 years of streamflow data and the increasing drought frequency amid an erratic monsoon.</p></div><div class="media_news_box">	<p class="p_media_title"><a href="https://www.livemint.com/opinion/online-views/saudi-arabia-pakistan-defence-agreement-smda-nuclear-umbrella-india-saudi-china-middle-east-west-asia-israel-qatar-war-11758521972077.html">Saudi Arabia and Pakistan just formed a defence pact—India must rethink strategy</a></p>	<p class="p_media_dida">2025-09-23T09:30:00(livemint) </p>	<p class="p_summary">This Nato-like defence commitment turns their ‘short-of-alliance partnership’ into a ‘formal bilateral alliance.’ For the KSA, Israel’s recent attack on Qatar and concerns for the Gulf region’s overall security is a major driver; whereas, for Pakistan, any development in its alliance network always had an Indian element as a motivating factor.</p></div><div class="media_news_box">	<p class="p_media_title"><a href="https://bdnews24.com/bangladesh/b6fb169e7a6a">Signals lowered at seaports, fresh low brews</a></p>	<p class="p_media_dida">2025-09-23T07:58:00(bdnews24) </p>	<p class="p_summary">The <span style="background-color:Yellow" class="highlight">Bangladesh</span> Meteorological Department has instructed the seaports of Chattogram, Cox’s Bazar, Mongla and Payra to lower their cautionary signals, saying there is currently no possibility of squally weather in the coastal areas. The directive came in a special bulletin issued by the department on....</p></div><div class="media_news_box">	<p class="p_media_title"><a href="https://www.brecorder.com/news/40384037/cm-meets-envoys-of-turkiye-bangladesh">CM meets envoys of Turkiye, <span style="background-color:Yellow" class="highlight">Bangladesh</span></a></p>	<p class="p_media_dida">2025-09-23T03:02:00(brecorder) </p>	<p class="p_summary">KARACHI: Sindh Chief Minister Syed Murad Ali Shah held separate meetings with the newly appointed Consul General of Turkiye Ergül Kadak, and the newly appointed Deputy High Commissioner of <span style="background-color:Yellow" class="highlight">Bangladesh</span> Muhammad Sakib Sadakat, at CM House. Welcoming both diplomats, the chief minister reaffirmed the....</p></div><div class="media_news_box">	<p class="p_media_title"><a href="https://dailytimes.com.pk/1372661/sindh-cm-turkish-and-bangladeshi-envoys-stress-trade-cooperation/">Sindh CM, Turkish and <span style="background-color:Yellow" class="highlight">Bangladesh</span>i envoys stress trade cooperation</a></p>	<p class="p_media_dida">2025-09-22T23:31:00(dailytimesPK) </p>	<p class="p_summary">Sindh Chief Minister Syed Murad Ali Shah on Tuesday held separate meetings with the newly appointed Consul General of Turkiye, Ergül Kadak and the newly appointed Deputy High Commissioner of <span style="background-color:Yellow" class="highlight">Bangladesh</span>, Muhammad Sakib Sadakat, at CM House. Welcoming both diplomats, the Chief Minister reaffirmed the....</p></div><div class="media_news_box">	<p class="p_media_title"><a href="https://reliefweb.int/report/bangladesh/wfp-bangladesh-country-brief-august-2025">WFP <span style="background-color:Yellow" class="highlight">Bangladesh</span> Country Brief, August 2025</a></p>	<p class="p_media_dida">2025-09-22T21:59:00(reliefWeb) </p>	<p class="p_summary">In Numbers 15,846 mt food distributed; USD 8.2 million distributed in cash-based transfers; USD 46.68 million net funding requirements under the country strategic plan (Sep 2025 – Feb 2026) 1.2 million people assisted* Operational Updates. Emergency preparedness.</p></div><div class="media_news_box">	<p class="p_media_title"><a href="https://reliefweb.int/report/bangladesh/wfp-bangladesh-country-brief-august-2025">WFP <span style="background-color:Yellow" class="highlight">Bangladesh</span> Country Brief, August 2025</a></p>	<p class="p_media_dida">2025-09-22T13:54:00(humanitarianresponse) </p>	<p class="p_summary">In Numbers 15,846 mt food distributed; USD 8.2 million distributed in cash-based transfers; USD 46.68 million net funding requirements under the country strategic plan (Sep 2025 Feb 2026) 1.2 million people assisted* Operational Updates. Emergency preparedness.</p></div><div class="media_news_box">	<p class="p_media_title"><a href="https://www.app.com.pk/domestic/cm-turkish-cg-bangladesh-deputy-hc-stress-trade-investment-cooperation/">CM, Turkish CG, <span style="background-color:Yellow" class="highlight">Bangladesh</span> Deputy HC, stress trade, investment cooperation</a></p>	<p class="p_media_dida">2025-09-22T13:08:00(app-en) </p>	<p class="p_summary">KARACHI, Sep 22 (APP): Sindh Chief Minister Syed Murad Ali Shah on Tuesday held separate meetings with the newly appointed Consul General of Turkiye, Ergül Kadak and the newly appointed Deputy High Commissioner of <span style="background-color:Yellow" class="highlight">Bangladesh</span>, Muhammad Sakib Sadakat, at CM House.</p></div><div class="media_news_box">	<p class="p_media_title"><a href="https://www.urdupoint.com/en/pakistan/cm-turkish-cg-bangladesh-deputy-hc-stress-2048701.html">CM, Turkish CG, <span style="background-color:Yellow" class="highlight">Bangladesh</span> Deputy HC, stress trade, investment cooperation</a></p>	<p class="p_media_dida">2025-09-22T12:48:00(urdupoint) </p>	<p class="p_summary">KARACHI, (UrduPoint / Pakistan Point News - 22nd Sep, 2025) Sindh Chief Minister Syed Murad Ali Shah on Tuesday held separate meetings with the newly appointed Consul General of Turkiye, Ergül Kadak and the newly appointed Deputy High Commissioner of <span style="background-color:Yellow" class="highlight">Bangladesh</span>, Muhammad Sakib Sadakat, at CM House.</p></div></div>
                    </div>
<div id="emmList"><div class="media_news_box">	<p class="p_media_title"><a href="https://www.cgiar.org/news-events/news/breakthrough-in-cold-tolerant-rice-development-benefits-haor-farmers-under-haor-kgf-project-in-bangladesh">Breakthrough in cold-tolerant rice development benefits haor farmers under Haor-KGF Project in <span style="background-color:Yellow" class="highlight">Bangladesh</span> International Rice Research Institute (IRRI) 17.09.25 Climate adaptation &amp; mitigation By Nusrat Jahan, Muhiuddin Faruquee, and Mohammad Rafiqul Islam <span style="background-color:Yellow" class="highlight">Bangladesh</span>, August 2025 – After fi… Read....</a></p>	<p class="p_media_dida">2025-09-23T20:27:00(cgiar) </p>	<p class="p_summary">By Nusrat Jahan, Muhiuddin Faruquee, and Mohammad Rafiqul Islam. <span style="background-color:Yellow" class="highlight">Bangladesh</span>, August 2025 – After five years of intensive research and collaboration, the Haor-KGF project has reached a major milestone: the development of advanced short-duration, cold-tolerant rice lines that can help farmers in....</p></div><div class="media_news_box">	<p class="p_media_title"><a href="https://www.news18.com/india/ganga-basin-faced-severe-drying-over-the-last-30-years-due-to-frequent-droughts-shows-new-study-ws-l-9591630.html">Ganga Basin Faced Severe Drying Over The Last 30 Years Due To Frequent Droughts, Shows New Study</a></p>	<p class="p_media_dida">2025-09-23T11:06:00(news18) </p>	<p class="p_summary">The Ganga River Basin, lifeline of over 600 million people, experienced its most severe drying trend from 1991 to 2020 – the worst in the past 1,000 years, according to a new research, which examined 1,300 years of streamflow data and the increasing drought frequency amid an erratic monsoon.</p></div><div class="media_news_box">	<p class="p_media_title"><a href="https://www.livemint.com/opinion/online-views/saudi-arabia-pakistan-defence-agreement-smda-nuclear-umbrella-india-saudi-china-middle-east-west-asia-israel-qatar-war-11758521972077.html">Saudi Arabia and Pakistan just formed a defence pact—India must rethink strategy</a></p>	<p class="p_media_dida">2025-09-23T09:30:00(livemint) </p>	<p class="p_summary">This Nato-like defence commitment turns their ‘short-of-alliance partnership’ into a ‘formal bilateral alliance.’ For the KSA, Israel’s recent attack on Qatar and concerns for the Gulf region’s overall security is a major driver; whereas, for Pakistan, any development in its alliance network always had an Indian element as a motivating factor.</p></div><div class="media_news_box">	<p class="p_media_title"><a href="https://bdnews24.com/bangladesh/b6fb169e7a6a">Signals lowered at seaports, fresh low brews</a></p>	<p class="p_media_dida">2025-09-23T07:58:00(bdnews24) </p>	<p class="p_summary">The <span style="background-color:Yellow" class="highlight">Bangladesh</span> Meteorological Department has instructed the seaports of Chattogram, Cox’s Bazar, Mongla and Payra to lower their cautionary signals, saying there is currently no possibility of squally weather in the coastal areas. The directive came in a special bulletin issued by the department on....</p></div><div class="media_news_box">	<p class="p_media_title"><a href="https://www.brecorder.com/news/40384037/cm-meets-envoys-of-turkiye-bangladesh">CM meets envoys of Turkiye, <span style="background-color:Yellow" class="highlight">Bangladesh</span></a></p>	<p class="p_media_dida">2025-09-23T03:02:00(brecorder) </p>	<p class="p_summary">KARACHI: Sindh Chief Minister Syed Murad Ali Shah held separate meetings with the newly appointed Consul General of Turkiye Ergül Kadak, and the newly appointed Deputy High Commissioner of <span style="background-color:Yellow" class="highlight">Bangladesh</span> Muhammad Sakib Sadakat, at CM House. Welcoming both diplomats, the chief minister reaffirmed the....</p></div><div class="media_news_box">	<p class="p_media_title"><a href="https://dailytimes.com.pk/1372661/sindh-cm-turkish-and-bangladeshi-envoys-stress-trade-cooperation/">Sindh CM, Turkish and <span style="background-color:Yellow" class="highlight">Bangladesh</span>i envoys stress trade cooperation</a></p>	<p class="p_media_dida">2025-09-22T23:31:00(dailytimesPK) </p>	<p class="p_summary">Sindh Chief Minister Syed Murad Ali Shah on Tuesday held separate meetings with the newly appointed Consul General of Turkiye, Ergül Kadak and the newly appointed Deputy High Commissioner of <span style="background-color:Yellow" class="highlight">Bangladesh</span>, Muhammad Sakib Sadakat, at CM House. Welcoming both diplomats, the Chief Minister reaffirmed the....</p></div><div class="media_news_box">	<p class="p_media_title"><a href="https://reliefweb.int/report/bangladesh/wfp-bangladesh-country-brief-august-2025">WFP <span style="background-color:Yellow" class="highlight">Bangladesh</span> Country Brief, August 2025</a></p>	<p class="p_media_dida">2025-09-22T21:59:00(reliefWeb) </p>	<p class="p_summary">In Numbers 15,846 mt food distributed; USD 8.2 million distributed in cash-based transfers; USD 46.68 million net funding requirements under the country strategic plan (Sep 2025 – Feb 2026) 1.2 million people assisted* Operational Updates. Emergency preparedness.</p></div><div class="media_news_box">	<p class="p_media_title"><a href="https://reliefweb.int/report/bangladesh/wfp-bangladesh-country-brief-august-2025">WFP <span style="background-color:Yellow" class="highlight">Bangladesh</span> Country Brief, August 2025</a></p>	<p class="p_media_dida">2025-09-22T13:54:00(humanitarianresponse) </p>	<p class="p_summary">In Numbers 15,846 mt food distributed; USD 8.2 million distributed in cash-based transfers; USD 46.68 million net funding requirements under the country strategic plan (Sep 2025 Feb 2026) 1.2 million people assisted* Operational Updates. Emergency preparedness.</p></div><div class="media_news_box">	<p class="p_media_title"><a href="https://www.app.com.pk/domestic/cm-turkish-cg-bangladesh-deputy-hc-stress-trade-investment-cooperation/">CM, Turkish CG, <span style="background-color:Yellow" class="highlight">Bangladesh</span> Deputy HC, stress trade, investment cooperation</a></p>	<p class="p_media_dida">2025-09-22T13:08:00(app-en) </p>	<p class="p_summary">KARACHI, Sep 22 (APP): Sindh Chief Minister Syed Murad Ali Shah on Tuesday held separate meetings with the newly appointed Consul General of Turkiye, Ergül Kadak and the newly appointed Deputy High Commissioner of <span style="background-color:Yellow" class="highlight">Bangladesh</span>, Muhammad Sakib Sadakat, at CM House.</p></div><div class="media_news_box">	<p class="p_media_title"><a href="https://www.urdupoint.com/en/pakistan/cm-turkish-cg-bangladesh-deputy-hc-stress-2048701.html">CM, Turkish CG, <span style="background-color:Yellow" class="highlight">Bangladesh</span> Deputy HC, stress trade, investment cooperation</a></p>	<p class="p_media_dida">2025-09-22T12:48:00(urdupoint) </p>	<p class="p_summary">KARACHI, (UrduPoint / Pakistan Point News - 22nd Sep, 2025) Sindh Chief Minister Syed Murad Ali Shah on Tuesday held separate meetings with the newly appointed Consul General of Turkiye, Ergül Kadak and the newly appointed Deputy High Commissioner of <span style="background-color:Yellow" class="highlight">Bangladesh</span>, Muhammad Sakib Sadakat, at CM House.</p></div></div>
<div class="media_news_box">	<p class="p_media_title"><a href="https://www.cgiar.org/news-events/news/breakthrough-in-cold-tolerant-rice-development-benefits-haor-farmers-under-haor-kgf-project-in-bangladesh">Breakthrough in cold-tolerant rice development benefits haor farmers under Haor-KGF Project in <span style="background-color:Yellow" class="highlight">Bangladesh</span> International Rice Research Institute (IRRI) 17.09.25 Climate adaptation &amp; mitigation By Nusrat Jahan, Muhiuddin Faruquee, and Mohammad Rafiqul Islam <span style="background-color:Yellow" class="highlight">Bangladesh</span>, August 2025 – After fi… Read....</a></p>	<p class="p_media_dida">2025-09-23T20:27:00(cgiar) </p>	<p class="p_summary">By Nusrat Jahan, Muhiuddin Faruquee, and Mohammad Rafiqul Islam. <span style="background-color:Yellow" class="highlight">Bangladesh</span>, August 2025 – After five years of intensive research and collaboration, the Haor-KGF project has reached a major milestone: the development of advanced short-duration, cold-tolerant rice lines that can help farmers in....</p></div>
<p class="p_media_title"><a href="https://www.cgiar.org/news-events/news/breakthrough-in-cold-tolerant-rice-development-benefits-haor-farmers-under-haor-kgf-project-in-bangladesh">Breakthrough in cold-tolerant rice development benefits haor farmers under Haor-KGF Project in <span style="background-color:Yellow" class="highlight">Bangladesh</span> International Rice Research Institute (IRRI) 17.09.25 Climate adaptation &amp; mitigation By Nusrat Jahan, Muhiuddin Faruquee, and Mohammad Rafiqul Islam <span style="background-color:Yellow" class="highlight">Bangladesh</span>, August 2025 – After fi… Read....</a></p>
<a href="https://www.cgiar.org/news-events/news/breakthrough-in-cold-tolerant-rice-development-benefits-haor-farmers-under-haor-kgf-project-in-bangladesh">Breakthrough in cold-tolerant rice development benefits haor farmers under Haor-KGF Project in <span style="background-color:Yellow" class="highlight">Bangladesh</span> International Rice Research Institute (IRRI) 17.09.25 Climate adaptation &amp; mitigation By Nusrat Jahan, Muhiuddin Faruquee, and Mohammad Rafiqul Islam <span style="background-color:Yellow" class="highlight">Bangladesh</span>, August 2025 – After fi… Read....</a>
<p class="p_media_title"><a href="https://www.cgiar.org/news-events/news/breakthrough-in-cold-tolerant-rice-development-benefits-haor-farmers-under-haor-kgf-project-in-bangladesh">Breakthrough in cold-tolerant rice development benefits haor farmers under Haor-KGF Project in <span style="background-color:Yellow" class="highlight">Bangladesh</span> International Rice Research Institute (IRRI) 17.09.25 Climate adaptation &amp; mitigation By Nusrat Jahan, Muhiuddin Faruquee, and Mohammad Rafiqul Islam <span style="background-color:Yellow" class="highlight">Bangladesh</span>, August 2025 – After fi… Read....</a></p>
<p class="p_media_dida">2025-09-23T20:27:00(cgiar) </p>
<p class="p_summary">By Nusrat Jahan, Muhiuddin Faruquee, and Mohammad Rafiqul Islam. <span style="background-color:Yellow" class="highlight">Bangladesh</span>, August 2025 – After five years of intensive research and collaboration, the Haor-KGF project has reached a major milestone: the development of advanced short-duration, cold-tolerant rice lines that can help farmers in....</p>
By Nusrat Jahan, Muhiuddin Faruquee, and Mohammad Rafiqul Islam. 
<span style="background-color:Yellow" class="highlight">Bangladesh</span>
, August 2025 – After five years of intensive research and collaboration, the Haor-KGF project has reached a major milestone: the development of advanced short-duration, cold-tolerant rice lines that can help farmers in....
<p class="p_summary">By Nusrat Jahan, Muhiuddin Faruquee, and Mohammad Rafiqul Islam. <span style="background-color:Yellow" class="highlight">Bangladesh</span>, August 2025 – After five years of intensive research and collaboration, the Haor-KGF project has reached a major milestone: the development of advanced short-duration, cold-tolerant rice lines that can help farmers in....</p>
<div class="media_news_box">	<p class="p_media_title"><a href="https://www.cgiar.org/news-events/news/breakthrough-in-cold-tolerant-rice-development-benefits-haor-farmers-under-haor-kgf-project-in-bangladesh">Breakthrough in cold-tolerant rice development benefits haor farmers under Haor-KGF Project in <span style="background-color:Yellow" class="highlight">Bangladesh</span> International Rice Research Institute (IRRI) 17.09.25 Climate adaptation &amp; mitigation By Nusrat Jahan, Muhiuddin Faruquee, and Mohammad Rafiqul Islam <span style="background-color:Yellow" class="highlight">Bangladesh</span>, August 2025 – After fi… Read....</a></p>	<p class="p_media_dida">2025-09-23T20:27:00(cgiar) </p>	<p class="p_summary">By Nusrat Jahan, Muhiuddin Faruquee, and Mohammad Rafiqul Islam. <span style="background-color:Yellow" class="highlight">Bangladesh</span>, August 2025 – After five years of intensive research and collaboration, the Haor-KGF project has reached a major milestone: the development of advanced short-duration, cold-tolerant rice lines that can help farmers in....</p></div>
<div class="media_news_box">	<p class="p_media_title"><a href="https://www.news18.com/india/ganga-basin-faced-severe-drying-over-the-last-30-years-due-to-frequent-droughts-shows-new-study-ws-l-9591630.html">Ganga Basin Faced Severe Drying Over The Last 30 Years Due To Frequent Droughts, Shows New Study</a></p>	<p class="p_media_dida">2025-09-23T11:06:00(news18) </p>	<p class="p_summary">The Ganga River Basin, lifeline of over 600 million people, experienced its most severe drying trend from 1991 to 2020 – the worst in the past 1,000 years, according to a new research, which examined 1,300 years of streamflow data and the increasing drought frequency amid an erratic monsoon.</p></div>
<div class="media_news_box">	<p class="p_media_title"><a href="https://www.livemint.com/opinion/online-views/saudi-arabia-pakistan-defence-agreement-smda-nuclear-umbrella-india-saudi-china-middle-east-west-asia-israel-qatar-war-11758521972077.html">Saudi Arabia and Pakistan just formed a defence pact—India must rethink strategy</a></p>	<p class="p_media_dida">2025-09-23T09:30:00(livemint) </p>	<p class="p_summary">This Nato-like defence commitment turns their ‘short-of-alliance partnership’ into a ‘formal bilateral alliance.’ For the KSA, Israel’s recent attack on Qatar and concerns for the Gulf region’s overall security is a major driver; whereas, for Pakistan, any development in its alliance network always had an Indian element as a motivating factor.</p></div>
<div class="media_news_box">	<p class="p_media_title"><a href="https://bdnews24.com/bangladesh/b6fb169e7a6a">Signals lowered at seaports, fresh low brews</a></p>	<p class="p_media_dida">2025-09-23T07:58:00(bdnews24) </p>	<p class="p_summary">The <span style="background-color:Yellow" class="highlight">Bangladesh</span> Meteorological Department has instructed the seaports of Chattogram, Cox’s Bazar, Mongla and Payra to lower their cautionary signals, saying there is currently no possibility of squally weather in the coastal areas. The directive came in a special bulletin issued by the department on....</p></div>
<div class="media_news_box">	<p class="p_media_title"><a href="https://www.brecorder.com/news/40384037/cm-meets-envoys-of-turkiye-bangladesh">CM meets envoys of Turkiye, <span style="background-color:Yellow" class="highlight">Bangladesh</span></a></p>	<p class="p_media_dida">2025-09-23T03:02:00(brecorder) </p>	<p class="p_summary">KARACHI: Sindh Chief Minister Syed Murad Ali Shah held separate meetings with the newly appointed Consul General of Turkiye Ergül Kadak, and the newly appointed Deputy High Commissioner of <span style="background-color:Yellow" class="highlight">Bangladesh</span> Muhammad Sakib Sadakat, at CM House. Welcoming both diplomats, the chief minister reaffirmed the....</p></div>
<div class="media_news_box">	<p class="p_media_title"><a href="https://dailytimes.com.pk/1372661/sindh-cm-turkish-and-bangladeshi-envoys-stress-trade-cooperation/">Sindh CM, Turkish and <span style="background-color:Yellow" class="highlight">Bangladesh</span>i envoys stress trade cooperation</a></p>	<p class="p_media_dida">2025-09-22T23:31:00(dailytimesPK) </p>	<p class="p_summary">Sindh Chief Minister Syed Murad Ali Shah on Tuesday held separate meetings with the newly appointed Consul General of Turkiye, Ergül Kadak and the newly appointed Deputy High Commissioner of <span style="background-color:Yellow" class="highlight">Bangladesh</span>, Muhammad Sakib Sadakat, at CM House. Welcoming both diplomats, the Chief Minister reaffirmed the....</p></div>
<div class="media_news_box">	<p class="p_media_title"><a href="https://reliefweb.int/report/bangladesh/wfp-bangladesh-country-brief-august-2025">WFP <span style="background-color:Yellow" class="highlight">Bangladesh</span> Country Brief, August 2025</a></p>	<p class="p_media_dida">2025-09-22T21:59:00(reliefWeb) </p>	<p class="p_summary">In Numbers 15,846 mt food distributed; USD 8.2 million distributed in cash-based transfers; USD 46.68 million net funding requirements under the country strategic plan (Sep 2025 – Feb 2026) 1.2 million people assisted* Operational Updates. Emergency preparedness.</p></div>
<div class="media_news_box">	<p class="p_media_title"><a href="https://reliefweb.int/report/bangladesh/wfp-bangladesh-country-brief-august-2025">WFP <span style="background-color:Yellow" class="highlight">Bangladesh</span> Country Brief, August 2025</a></p>	<p class="p_media_dida">2025-09-22T13:54:00(humanitarianresponse) </p>	<p class="p_summary">In Numbers 15,846 mt food distributed; USD 8.2 million distributed in cash-based transfers; USD 46.68 million net funding requirements under the country strategic plan (Sep 2025 Feb 2026) 1.2 million people assisted* Operational Updates. Emergency preparedness.</p></div>
<div class="media_news_box">	<p class="p_media_title"><a href="https://www.app.com.pk/domestic/cm-turkish-cg-bangladesh-deputy-hc-stress-trade-investment-cooperation/">CM, Turkish CG, <span style="background-color:Yellow" class="highlight">Bangladesh</span> Deputy HC, stress trade, investment cooperation</a></p>	<p class="p_media_dida">2025-09-22T13:08:00(app-en) </p>	<p class="p_summary">KARACHI, Sep 22 (APP): Sindh Chief Minister Syed Murad Ali Shah on Tuesday held separate meetings with the newly appointed Consul General of Turkiye, Ergül Kadak and the newly appointed Deputy High Commissioner of <span style="background-color:Yellow" class="highlight">Bangladesh</span>, Muhammad Sakib Sadakat, at CM House.</p></div>
<div class="media_news_box">	<p class="p_media_title"><a href="https://www.urdupoint.com/en/pakistan/cm-turkish-cg-bangladesh-deputy-hc-stress-2048701.html">CM, Turkish CG, <span style="background-color:Yellow" class="highlight">Bangladesh</span> Deputy HC, stress trade, investment cooperation</a></p>	<p class="p_media_dida">2025-09-22T12:48:00(urdupoint) </p>	<p class="p_summary">KARACHI, (UrduPoint / Pakistan Point News - 22nd Sep, 2025) Sindh Chief Minister Syed Murad Ali Shah on Tuesday held separate meetings with the newly appointed Consul General of Turkiye, Ergül Kadak and the newly appointed Deputy High Commissioner of <span style="background-color:Yellow" class="highlight">Bangladesh</span>, Muhammad Sakib Sadakat, at CM House.</p></div>
<div id="emmList"><div class="media_news_box">	<p class="p_media_title"><a href="https://www.cgiar.org/news-events/news/breakthrough-in-cold-tolerant-rice-development-benefits-haor-farmers-under-haor-kgf-project-in-bangladesh">Breakthrough in cold-tolerant rice development benefits haor farmers under Haor-KGF Project in <span style="background-color:Yellow" class="highlight">Bangladesh</span> International Rice Research Institute (IRRI) 17.09.25 Climate adaptation &amp; mitigation By Nusrat Jahan, Muhiuddin Faruquee, and Mohammad Rafiqul Islam <span style="background-color:Yellow" class="highlight">Bangladesh</span>, August 2025 – After fi… Read....</a></p>	<p class="p_media_dida">2025-09-23T20:27:00(cgiar) </p>	<p class="p_summary">By Nusrat Jahan, Muhiuddin Faruquee, and Mohammad Rafiqul Islam. <span style="background-color:Yellow" class="highlight">Bangladesh</span>, August 2025 – After five years of intensive research and collaboration, the Haor-KGF project has reached a major milestone: the development of advanced short-duration, cold-tolerant rice lines that can help farmers in....</p></div><div class="media_news_box">	<p class="p_media_title"><a href="https://www.news18.com/india/ganga-basin-faced-severe-drying-over-the-last-30-years-due-to-frequent-droughts-shows-new-study-ws-l-9591630.html">Ganga Basin Faced Severe Drying Over The Last 30 Years Due To Frequent Droughts, Shows New Study</a></p>	<p class="p_media_dida">2025-09-23T11:06:00(news18) </p>	<p class="p_summary">The Ganga River Basin, lifeline of over 600 million people, experienced its most severe drying trend from 1991 to 2020 – the worst in the past 1,000 years, according to a new research, which examined 1,300 years of streamflow data and the increasing drought frequency amid an erratic monsoon.</p></div><div class="media_news_box">	<p class="p_media_title"><a href="https://www.livemint.com/opinion/online-views/saudi-arabia-pakistan-defence-agreement-smda-nuclear-umbrella-india-saudi-china-middle-east-west-asia-israel-qatar-war-11758521972077.html">Saudi Arabia and Pakistan just formed a defence pact—India must rethink strategy</a></p>	<p class="p_media_dida">2025-09-23T09:30:00(livemint) </p>	<p class="p_summary">This Nato-like defence commitment turns their ‘short-of-alliance partnership’ into a ‘formal bilateral alliance.’ For the KSA, Israel’s recent attack on Qatar and concerns for the Gulf region’s overall security is a major driver; whereas, for Pakistan, any development in its alliance network always had an Indian element as a motivating factor.</p></div><div class="media_news_box">	<p class="p_media_title"><a href="https://bdnews24.com/bangladesh/b6fb169e7a6a">Signals lowered at seaports, fresh low brews</a></p>	<p class="p_media_dida">2025-09-23T07:58:00(bdnews24) </p>	<p class="p_summary">The <span style="background-color:Yellow" class="highlight">Bangladesh</span> Meteorological Department has instructed the seaports of Chattogram, Cox’s Bazar, Mongla and Payra to lower their cautionary signals, saying there is currently no possibility of squally weather in the coastal areas. The directive came in a special bulletin issued by the department on....</p></div><div class="media_news_box">	<p class="p_media_title"><a href="https://www.brecorder.com/news/40384037/cm-meets-envoys-of-turkiye-bangladesh">CM meets envoys of Turkiye, <span style="background-color:Yellow" class="highlight">Bangladesh</span></a></p>	<p class="p_media_dida">2025-09-23T03:02:00(brecorder) </p>	<p class="p_summary">KARACHI: Sindh Chief Minister Syed Murad Ali Shah held separate meetings with the newly appointed Consul General of Turkiye Ergül Kadak, and the newly appointed Deputy High Commissioner of <span style="background-color:Yellow" class="highlight">Bangladesh</span> Muhammad Sakib Sadakat, at CM House. Welcoming both diplomats, the chief minister reaffirmed the....</p></div><div class="media_news_box">	<p class="p_media_title"><a href="https://dailytimes.com.pk/1372661/sindh-cm-turkish-and-bangladeshi-envoys-stress-trade-cooperation/">Sindh CM, Turkish and <span style="background-color:Yellow" class="highlight">Bangladesh</span>i envoys stress trade cooperation</a></p>	<p class="p_media_dida">2025-09-22T23:31:00(dailytimesPK) </p>	<p class="p_summary">Sindh Chief Minister Syed Murad Ali Shah on Tuesday held separate meetings with the newly appointed Consul General of Turkiye, Ergül Kadak and the newly appointed Deputy High Commissioner of <span style="background-color:Yellow" class="highlight">Bangladesh</span>, Muhammad Sakib Sadakat, at CM House. Welcoming both diplomats, the Chief Minister reaffirmed the....</p></div><div class="media_news_box">	<p class="p_media_title"><a href="https://reliefweb.int/report/bangladesh/wfp-bangladesh-country-brief-august-2025">WFP <span style="background-color:Yellow" class="highlight">Bangladesh</span> Country Brief, August 2025</a></p>	<p class="p_media_dida">2025-09-22T21:59:00(reliefWeb) </p>	<p class="p_summary">In Numbers 15,846 mt food distributed; USD 8.2 million distributed in cash-based transfers; USD 46.68 million net funding requirements under the country strategic plan (Sep 2025 – Feb 2026) 1.2 million people assisted* Operational Updates. Emergency preparedness.</p></div><div class="media_news_box">	<p class="p_media_title"><a href="https://reliefweb.int/report/bangladesh/wfp-bangladesh-country-brief-august-2025">WFP <span style="background-color:Yellow" class="highlight">Bangladesh</span> Country Brief, August 2025</a></p>	<p class="p_media_dida">2025-09-22T13:54:00(humanitarianresponse) </p>	<p class="p_summary">In Numbers 15,846 mt food distributed; USD 8.2 million distributed in cash-based transfers; USD 46.68 million net funding requirements under the country strategic plan (Sep 2025 Feb 2026) 1.2 million people assisted* Operational Updates. Emergency preparedness.</p></div><div class="media_news_box">	<p class="p_media_title"><a href="https://www.app.com.pk/domestic/cm-turkish-cg-bangladesh-deputy-hc-stress-trade-investment-cooperation/">CM, Turkish CG, <span style="background-color:Yellow" class="highlight">Bangladesh</span> Deputy HC, stress trade, investment cooperation</a></p>	<p class="p_media_dida">2025-09-22T13:08:00(app-en) </p>	<p class="p_summary">KARACHI, Sep 22 (APP): Sindh Chief Minister Syed Murad Ali Shah on Tuesday held separate meetings with the newly appointed Consul General of Turkiye, Ergül Kadak and the newly appointed Deputy High Commissioner of <span style="background-color:Yellow" class="highlight">Bangladesh</span>, Muhammad Sakib Sadakat, at CM House.</p></div><div class="media_news_box">	<p class="p_media_title"><a href="https://www.urdupoint.com/en/pakistan/cm-turkish-cg-bangladesh-deputy-hc-stress-2048701.html">CM, Turkish CG, <span style="background-color:Yellow" class="highlight">Bangladesh</span> Deputy HC, stress trade, investment cooperation</a></p>	<p class="p_media_dida">2025-09-22T12:48:00(urdupoint) </p>	<p class="p_summary">KARACHI, (UrduPoint / Pakistan Point News - 22nd Sep, 2025) Sindh Chief Minister Syed Murad Ali Shah on Tuesday held separate meetings with the newly appointed Consul General of Turkiye, Ergül Kadak and the newly appointed Deputy High Commissioner of <span style="background-color:Yellow" class="highlight">Bangladesh</span>, Muhammad Sakib Sadakat, at CM House.</p></div></div>"""

# save outputs as df, then to deep_tab_outputs/media.csv

'<div id="contentSectionNews" class="col-lg-12">\n                    <h2>All headlines on this Alert</h2>\n                    <p class="p_summary">The headlines below have been automatically extracted by the <a href="http://emm.newsbrief.eu" "target="_blank">Europe Media Monitor</a>.</p>\n                    <div id="scrollList" class="alert_summary_fifth_b" style="max-height:500px">\n                        <div id="emmList"><div class="media_news_box">\t<p class="p_media_title"><a href="https://www.cgiar.org/news-events/news/breakthrough-in-cold-tolerant-rice-development-benefits-haor-farmers-under-haor-kgf-project-in-bangladesh">Breakthrough in cold-tolerant rice development benefits haor farmers under Haor-KGF Project in <span style="background-color:Yellow" class="highlight">Bangladesh</span> International Rice Research Institute (IRRI) 17.09.25 Climate adaptation &amp; mitigation By Nusrat Jahan, Muhiuddin Faruquee, and Mohammad Rafiqul Islam <span style="background-color:Yello

In [ ]:
# do an inner join on data/gdacs_df_20260225_215447.csv, summary.csv, media.csv, impact.csv so all of the info is in one csv file, then save as data/outputs/20260225_215447.csv